# 21. 取引量と平均エクスポージャーを揃えた比較
出典: FX (2).ipynb、セルindex [44, 45]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 44


In [ ]:
# ============================================================
# USD/JPY
# NESTED POSITION SIZING TEST
#
# 目的:
# Calibration済みConfidenceを利用したPosition Sizingが
# FIXED 1.0xよりOOSで価値を持つか検証する。
#
#
# 固定するもの:
#
# ・15分足
# ・Random Forest
# ・Probability Calibration
# ・Confidence Threshold
# ・Session Filter
# ・30分固定Exit
# ・Transaction Cost
#
#
# 今回変更するもの:
#
# ・Position Sizeのみ
#
#
# 非常に重要:
#
# Test年を見てSizingを選ばない。
#
# Train
#   ↓
# Calibration
#   ↓
# Validation
#   ↓
# Threshold / Session
#   ↓
# Position Sizing選択
#   ↓
# Test
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. 前コードの確認
# ============================================================

REQUIRED = [

    "data",
    "FEATURES",

    "build_model",
    "predict_frame",

    "choose_threshold",
    "choose_session",
    "select_trades",

    "fit_calibrators",
    "choose_calibrator",
    "apply_probability",

    "BASE_COST",

    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",
]


missing = [

    name

    for name in REQUIRED

    if name not in globals()
]


if missing:

    raise RuntimeError(

        "前のCalibrationコードを先に実行してください。\n"
        f"不足: {missing}"

    )


# ============================================================
# 1. CONFIG
# ============================================================

# ------------------------------------------------------------
# Sizing研究用の相対サイズ上限
#
# これは本番レバレッジではない。
#
# あくまで、
#
# 1.0 = 基準サイズ
# 1.5 = 基準の1.5倍
#
# という相対値。
# ------------------------------------------------------------

MIN_RELATIVE_SIZE = 0.25

MAX_RELATIVE_SIZE = 2.00


# ------------------------------------------------------------
# Position Sizing候補
#
# 候補数を意図的に少なくしている。
#
# 候補を大量に作るとValidation過学習するため。
# ------------------------------------------------------------

SIZING_POLICIES = [

    "FIXED",
    "GENTLE",
    "MODERATE",
    "STRONG",

]


# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

BOOTSTRAP_ITERATIONS = 5000

BOOTSTRAP_BLOCK_DAYS = 20

RANDOM_SEED = 42


# ------------------------------------------------------------
# 保存先
# ------------------------------------------------------------

OUTPUT_DIR = (

    Path.cwd()

    /

    (
        "nested_position_sizing_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. Confidence → Raw Position Size
# ============================================================

def raw_position_size(
    confidence,
    policy,
):

    """
    Confidenceから相対Position Sizeを返す。

    FIXED:
        常に1.0

    GENTLE:
        ゆるやかにサイズ変更

    MODERATE:
        中程度

    STRONG:
        Confidence依存を強くする
    """

    confidence = np.asarray(
        confidence,
        dtype=float
    )


    if policy == "FIXED":

        return np.ones(
            len(confidence)
        )


    if policy == "GENTLE":

        return np.select(

            [

                confidence < 0.56,

                confidence < 0.58,

                confidence < 0.60,

                confidence < 0.62,

                confidence < 0.65,

                confidence >= 0.65,

            ],

            [

                0.75,

                0.90,

                1.05,

                1.15,

                1.30,

                1.40,

            ],

            default=1.0,

        )


    if policy == "MODERATE":

        return np.select(

            [

                confidence < 0.56,

                confidence < 0.58,

                confidence < 0.60,

                confidence < 0.62,

                confidence < 0.65,

                confidence >= 0.65,

            ],

            [

                0.50,

                0.75,

                1.00,

                1.20,

                1.40,

                1.50,

            ],

            default=1.0,

        )


    if policy == "STRONG":

        return np.select(

            [

                confidence < 0.56,

                confidence < 0.58,

                confidence < 0.60,

                confidence < 0.62,

                confidence < 0.65,

                confidence >= 0.65,

            ],

            [

                0.35,

                0.60,

                0.90,

                1.20,

                1.50,

                1.50,

            ],

            default=1.0,

        )


    raise ValueError(
        f"Unknown sizing policy: {policy}"
    )


# ============================================================
# 3. ValidationでExposureを正規化
# ============================================================

def fit_sizing_scale(
    confidence,
    policy,
):

    """
    Validation上で平均Raw Sizeを計算。

    平均Position Sizeが約1になるようにScaleを作る。

    これにより、

        Position Sizingで儲かった

    のか、

        単純に平均レバレッジを増やした

    のかを分離する。
    """

    raw_size = raw_position_size(

        confidence,
        policy,

    )


    mean_size = np.mean(
        raw_size
    )


    if (
        not np.isfinite(
            mean_size
        )

        or

        mean_size <= 0
    ):

        return 1.0


    return (
        1.0
        /
        mean_size
    )


# ============================================================
# 4. 実際のPosition Size
# ============================================================

def apply_sizing_policy(
    confidence,
    policy,
    validation_scale,
):

    raw_size = raw_position_size(

        confidence,
        policy,

    )


    final_size = (

        raw_size

        *

        validation_scale

    )


    final_size = np.clip(

        final_size,

        MIN_RELATIVE_SIZE,

        MAX_RELATIVE_SIZE,

    )


    return final_size


# ============================================================
# 5. Return Stats
# ============================================================

def return_stats(
    returns,
):

    """
    Sized Returnの評価。

    Returnはdecimal前提。

    例:
    0.001 = +0.1%
    """

    r = (

        pd.Series(
            returns
        )

        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )

        .dropna()

        .astype(float)

        .to_numpy()

    )


    n = len(
        r
    )


    if n == 0:

        return {

            "trades": 0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "growth":
                np.nan,

            "max_dd":
                np.nan,

            "trade_sharpe":
                np.nan,

            "return_to_dd":
                np.nan,
        }


    # --------------------------------------------------------
    # Win rate
    # --------------------------------------------------------

    win_rate = np.mean(
        r > 0
    )


    # --------------------------------------------------------
    # PF
    # --------------------------------------------------------

    gross_profit = r[
        r > 0
    ].sum()


    gross_loss = -r[
        r < 0
    ].sum()


    if gross_loss > 0:

        profit_factor = (

            gross_profit

            /

            gross_loss
        )

    elif gross_profit > 0:

        profit_factor = np.inf

    else:

        profit_factor = np.nan


    # --------------------------------------------------------
    # Equity Curve
    # --------------------------------------------------------

    equity = np.cumprod(
        1.0 + r
    )


    equity_with_start = np.concatenate(

        [

            np.array(
                [1.0]
            ),

            equity,

        ]

    )


    running_peak = np.maximum.accumulate(

        equity_with_start

    )


    drawdown = (

        equity_with_start

        /

        running_peak

        -

        1.0

    )


    max_dd = float(
        np.min(
            drawdown
        )
    )


    growth = float(

        equity[
            -1
        ]

        -

        1.0

    )


    # --------------------------------------------------------
    # Trade-level Sharpe-like
    #
    # 真の年率Sharpeではない。
    #
    # 同一戦略間比較用。
    # --------------------------------------------------------

    std = np.std(
        r,
        ddof=1
    )


    if (
        np.isfinite(
            std
        )

        and

        std > 0
    ):

        trade_sharpe = (

            np.mean(
                r
            )

            /

            std

            *

            np.sqrt(
                n
            )
        )

    else:

        trade_sharpe = np.nan


    # --------------------------------------------------------
    # Return / DD
    # --------------------------------------------------------

    if (
        max_dd < 0
        and
        np.isfinite(
            growth
        )
    ):

        return_to_dd = (

            growth

            /

            abs(
                max_dd
            )
        )

    else:

        return_to_dd = np.nan


    return {

        "trades":
            n,

        "win_rate":
            win_rate,

        "avg_return":
            float(
                np.mean(
                    r
                )
            ),

        "median_return":
            float(
                np.median(
                    r
                )
            ),

        "profit_factor":
            float(
                profit_factor
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "trade_sharpe":
            float(
                trade_sharpe
            ),

        "return_to_dd":
            float(
                return_to_dd
            ),
    }


# ============================================================
# 6. ValidationでSizing Policy比較
# ============================================================

def evaluate_sizing_candidates(
    validation_trades,
):

    if len(
        validation_trades
    ) == 0:

        return (
            None,
            pd.DataFrame()
        )


    if "confidence" not in validation_trades.columns:

        raise RuntimeError(
            "validation_tradesにconfidenceがありません。"
        )


    if "net_return" not in validation_trades.columns:

        raise RuntimeError(
            "validation_tradesにnet_returnがありません。"
        )


    rows = []


    for policy in SIZING_POLICIES:


        # ----------------------------------------------------
        # ValidationだけからScaleを決める
        # ----------------------------------------------------

        scale = fit_sizing_scale(

            validation_trades[
                "confidence"
            ],

            policy,

        )


        size = apply_sizing_policy(

            validation_trades[
                "confidence"
            ],

            policy,

            scale,

        )


        sized_return = (

            validation_trades[
                "net_return"
            ].to_numpy(
                dtype=float
            )

            *

            size

        )


        stats = return_stats(
            sized_return
        )


        rows.append({

            "policy":
                policy,

            "validation_scale":
                scale,

            "mean_size":
                np.mean(
                    size
                ),

            "min_size":
                np.min(
                    size
                ),

            "max_size":
                np.max(
                    size
                ),

            **stats,

        })


    table = pd.DataFrame(
        rows
    )


    # ========================================================
    # POLICY SELECTION
    #
    # PFだけでは選ばない。
    #
    # 条件:
    #
    # avg_return > 0
    # PF > 1
    #
    # の候補から
    #
    # Return / DD
    #
    # を最優先する。
    #
    # ========================================================

    eligible = table.loc[

        (
            table[
                "avg_return"
            ]
            >
            0
        )

        &

        (
            table[
                "profit_factor"
            ]
            >
            1
        )

        &

        np.isfinite(
            table[
                "return_to_dd"
            ]
        )

    ].copy()


    if len(
        eligible
    ) == 0:

        selected_policy = (
            "FIXED"
        )

    else:

        eligible = eligible.sort_values(

            [

                "return_to_dd",
                "profit_factor",
                "avg_return",

            ],

            ascending=[

                False,
                False,
                False,

            ]

        )


        selected_policy = (

            eligible
            .iloc[0]
            ["policy"]

        )


    selected_row = (

        table.loc[
            table[
                "policy"
            ]
            ==
            selected_policy
        ]

        .iloc[0]

    )


    return (
        selected_row,
        table,
    )


# ============================================================
# 7. Walk Forward
# ============================================================

years = sorted(
    data.index.year.unique()
)


annual_rows = []

validation_policy_tables = []

test_trade_frames = []


for test_year in years:


    validation_year = (
        test_year
        -
        1
    )


    previous_years = [

        y

        for y in years

        if y
        <
        validation_year

    ]


    if (
        len(
            previous_years
        )
        <
        MIN_TRAIN_YEARS
    ):

        continue


    if (
        validation_year
        not in years
    ):

        continue


    validation_start = pd.Timestamp(

        year=
            validation_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_start = pd.Timestamp(

        year=
            test_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_end = pd.Timestamp(

        year=
            test_year + 1,

        month=1,

        day=1,

        tz="UTC",

    )


    # ========================================================
    # TRAIN
    # ========================================================

    train = data.loc[

        (
            data.index
            <
            validation_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            validation_start
        )

    ].copy()


    # ========================================================
    # VALIDATION
    # ========================================================

    validation = data.loc[

        (
            data.index
            >=
            validation_start
        )

        &

        (
            data.index
            <
            test_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_start
        )

    ].copy()


    # ========================================================
    # FINAL TRAIN
    # ========================================================

    final_train = data.loc[

        (
            data.index
            <
            test_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_start
        )

    ].copy()


    # ========================================================
    # TEST
    # ========================================================

    test = data.loc[

        (
            data.index
            >=
            test_start
        )

        &

        (
            data.index
            <
            test_end
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_end
        )

    ].copy()


    if (

        len(train)
        <
        MIN_TRAIN_ROWS

        or

        len(validation)
        <
        MIN_EVAL_ROWS

        or

        len(final_train)
        <
        MIN_TRAIN_ROWS

        or

        len(test)
        <
        MIN_EVAL_ROWS

    ):

        continue


    print()
    print("=" * 70)

    print(
        "POSITION SIZING TEST YEAR",
        test_year
    )

    print("=" * 70)


    # ========================================================
    # 8. Validation Model
    # ========================================================

    model = build_model()


    model.fit(

        train[
            FEATURES
        ],

        train[
            "target"
        ]

    )


    validation_raw = predict_frame(

        model,

        validation,

    )


    # ========================================================
    # 9. Calibration
    # ========================================================

    calibrators, train_oof = (

        fit_calibrators(
            train
        )

    )


    selected_calibration, calibration_table = (

        choose_calibrator(

            calibrators,

            validation_raw[
                "p_up"
            ],

            validation[
                "target"
            ],

        )

    )


    print(
        "Calibration:",
        selected_calibration
    )


    validation_probability = (

        calibrators[
            selected_calibration
        ]

        .predict(

            validation_raw[
                "p_up"
            ]

        )

    )


    validation_pred = apply_probability(

        validation_raw,

        validation_probability,

    )


    # ========================================================
    # 10. Threshold
    # ========================================================

    threshold_choice, _ = (

        choose_threshold(

            validation_pred

        )

    )


    if threshold_choice is None:

        print(
            "Threshold selection failed."
        )

        continue


    threshold = float(

        threshold_choice[
            "threshold"
        ]

    )


    # ========================================================
    # 11. Session
    # ========================================================

    session_choice, _ = (

        choose_session(

            validation_pred,

            threshold,

        )

    )


    if session_choice is None:

        print(
            "Session selection failed."
        )

        continue


    session_policy = (

        session_choice[
            "session_policy"
        ]

    )


    print(
        "Threshold:",
        threshold
    )


    print(
        "Session:",
        session_policy
    )


    # ========================================================
    # 12. Validation Trades
    # ========================================================

    validation_trades = select_trades(

        validation_pred,

        threshold=
            threshold,

        session_policy=
            session_policy,

        cost=
            BASE_COST,

    )


    if len(
        validation_trades
    ) == 0:

        print(
            "Validation trades = 0"
        )

        continue


    # ========================================================
    # 13. Position SizingをValidationだけで選ぶ
    # ========================================================

    sizing_choice, sizing_table = (

        evaluate_sizing_candidates(

            validation_trades

        )

    )


    if sizing_choice is None:

        continue


    chosen_policy = (

        sizing_choice[
            "policy"
        ]

    )


    sizing_scale = float(

        sizing_choice[
            "validation_scale"
        ]

    )


    sizing_table[
        "test_year"
    ] = (
        test_year
    )


    sizing_table[
        "validation_year"
    ] = (
        validation_year
    )


    validation_policy_tables.append(

        sizing_table

    )


    print()
    print(
        "VALIDATION SIZING CANDIDATES"
    )

    print(
        sizing_table[
            [

                "policy",
                "mean_size",
                "avg_return",
                "profit_factor",
                "growth",
                "max_dd",
                "return_to_dd",

            ]
        ]

        .to_string(
            index=False
        )

    )


    print()
    print(
        "Selected Sizing:",
        chosen_policy
    )


    print(
        "Validation Scale:",
        sizing_scale
    )


    # ========================================================
    # 14. TEST用Calibration
    # ========================================================

    test_calibrators, final_oof = (

        fit_calibrators(

            final_train

        )

    )


    if (

        selected_calibration

        in

        test_calibrators

    ):

        test_calibration = (

            selected_calibration

        )

    else:

        test_calibration = (
            "RAW"
        )


    # ========================================================
    # 15. Final Model
    # ========================================================

    final_model = build_model()


    final_model.fit(

        final_train[
            FEATURES
        ],

        final_train[
            "target"
        ]

    )


    test_raw = predict_frame(

        final_model,

        test,

    )


    test_probability = (

        test_calibrators[
            test_calibration
        ]

        .predict(

            test_raw[
                "p_up"
            ]

        )

    )


    test_pred = apply_probability(

        test_raw,

        test_probability,

    )


    # ========================================================
    # 16. Test Trades
    #
    # FIXEDとSIZEDで同じEntryを使う
    # ========================================================

    test_trades = select_trades(

        test_pred,

        threshold=
            threshold,

        session_policy=
            session_policy,

        cost=
            BASE_COST,

    )


    if len(
        test_trades
    ) == 0:

        print(
            "Test trades = 0"
        )

        continue


    # ========================================================
    # 17. FIXED RETURN
    # ========================================================

    fixed_return = (

        test_trades[
            "net_return"
        ]

        .to_numpy(
            dtype=float
        )

    )


    # ========================================================
    # 18. TESTのPosition Size
    #
    # ScaleはValidationで決定済み。
    #
    # TestデータでScaleを再調整しない。
    # ========================================================

    test_size = apply_sizing_policy(

        test_trades[
            "confidence"
        ],

        chosen_policy,

        sizing_scale,

    )


    sized_return = (

        fixed_return

        *

        test_size

    )


    # ========================================================
    # 19. Statistics
    # ========================================================

    fixed_stats = return_stats(

        fixed_return

    )


    sized_stats = return_stats(

        sized_return

    )


    print()
    print(
        "TEST RESULT"
    )


    print(
        "Trades:",
        len(
            test_trades
        )
    )


    print(
        "Mean Size:",
        np.mean(
            test_size
        )
    )


    print()


    print(
        "FIXED PF:",
        fixed_stats[
            "profit_factor"
        ]
    )


    print(
        "SIZED PF:",
        sized_stats[
            "profit_factor"
        ]
    )


    print()


    print(
        "FIXED Avg:",
        fixed_stats[
            "avg_return"
        ]
        *
        100,
        "%"
    )


    print(
        "SIZED Avg:",
        sized_stats[
            "avg_return"
        ]
        *
        100,
        "%"
    )


    print()


    print(
        "FIXED Growth:",
        fixed_stats[
            "growth"
        ]
        *
        100,
        "%"
    )


    print(
        "SIZED Growth:",
        sized_stats[
            "growth"
        ]
        *
        100,
        "%"
    )


    print()


    print(
        "FIXED Max DD:",
        fixed_stats[
            "max_dd"
        ]
        *
        100,
        "%"
    )


    print(
        "SIZED Max DD:",
        sized_stats[
            "max_dd"
        ]
        *
        100,
        "%"
    )


    # ========================================================
    # 20. Trade-level保存
    # ========================================================

    temp = test_trades.copy()


    temp[
        "test_year"
    ] = (
        test_year
    )


    temp[
        "calibration_method"
    ] = (
        test_calibration
    )


    temp[
        "threshold"
    ] = (
        threshold
    )


    temp[
        "session_policy"
    ] = (
        session_policy
    )


    temp[
        "sizing_policy"
    ] = (
        chosen_policy
    )


    temp[
        "position_size"
    ] = (
        test_size
    )


    temp[
        "fixed_return"
    ] = (
        fixed_return
    )


    temp[
        "sized_return"
    ] = (
        sized_return
    )


    test_trade_frames.append(
        temp
    )


    # ========================================================
    # 21. Annual result
    # ========================================================

    annual_rows.append({

        "test_year":
            test_year,

        "calibration":
            test_calibration,

        "threshold":
            threshold,

        "session":
            session_policy,

        "sizing_policy":
            chosen_policy,

        "trades":
            len(
                test_trades
            ),

        "mean_size":
            np.mean(
                test_size
            ),

        "fixed_avg":
            fixed_stats[
                "avg_return"
            ],

        "sized_avg":
            sized_stats[
                "avg_return"
            ],

        "fixed_pf":
            fixed_stats[
                "profit_factor"
            ],

        "sized_pf":
            sized_stats[
                "profit_factor"
            ],

        "fixed_growth":
            fixed_stats[
                "growth"
            ],

        "sized_growth":
            sized_stats[
                "growth"
            ],

        "fixed_dd":
            fixed_stats[
                "max_dd"
            ],

        "sized_dd":
            sized_stats[
                "max_dd"
            ],

        "fixed_sharpe":
            fixed_stats[
                "trade_sharpe"
            ],

        "sized_sharpe":
            sized_stats[
                "trade_sharpe"
            ],

        "fixed_return_dd":
            fixed_stats[
                "return_to_dd"
            ],

        "sized_return_dd":
            sized_stats[
                "return_to_dd"
            ],

    })


# ============================================================
# 22. Annual Results
# ============================================================

annual_results = pd.DataFrame(

    annual_rows

)


if annual_results.empty:

    raise RuntimeError(

        "Position Sizing評価年が作れませんでした。"

    )


print()
print("=" * 80)

print(
    "ANNUAL POSITION SIZING RESULTS"
)

print("=" * 80)


annual_show = annual_results.copy()


for col in [

    "fixed_avg",
    "sized_avg",

    "fixed_growth",
    "sized_growth",

    "fixed_dd",
    "sized_dd",

]:

    annual_show[
        col
    ] *= 100


print(

    annual_show.to_string(
        index=False
    )

)


# ============================================================
# 23. Policy Selection Frequency
# ============================================================

print()
print("=" * 80)

print(
    "SIZING POLICY SELECTION FREQUENCY"
)

print("=" * 80)


print(

    annual_results[
        "sizing_policy"
    ]

    .value_counts()

)


# ============================================================
# 24. Year Stability
# ============================================================

annual_results[
    "avg_better"
] = (

    annual_results[
        "sized_avg"
    ]

    >

    annual_results[
        "fixed_avg"
    ]

)


annual_results[
    "pf_better"
] = (

    annual_results[
        "sized_pf"
    ]

    >

    annual_results[
        "fixed_pf"
    ]

)


annual_results[
    "growth_better"
] = (

    annual_results[
        "sized_growth"
    ]

    >

    annual_results[
        "fixed_growth"
    ]

)


annual_results[
    "dd_better"
] = (

    annual_results[
        "sized_dd"
    ]

    >

    annual_results[
        "fixed_dd"
    ]

)


annual_results[
    "return_dd_better"
] = (

    annual_results[
        "sized_return_dd"
    ]

    >

    annual_results[
        "fixed_return_dd"
    ]

)


n_years = len(
    annual_results
)


print()
print("=" * 80)

print(
    "POSITION SIZING YEAR STABILITY"
)

print("=" * 80)


print(
    "Evaluated years:",
    n_years
)


print(
    "Avg Return better:",
    annual_results[
        "avg_better"
    ].sum(),
    "/",
    n_years
)


print(
    "PF better:",
    annual_results[
        "pf_better"
    ].sum(),
    "/",
    n_years
)


print(
    "Growth better:",
    annual_results[
        "growth_better"
    ].sum(),
    "/",
    n_years
)


print(
    "Max DD better:",
    annual_results[
        "dd_better"
    ].sum(),
    "/",
    n_years
)


print(
    "Return/DD better:",
    annual_results[
        "return_dd_better"
    ].sum(),
    "/",
    n_years
)


# ============================================================
# 25. Overall OOS
# ============================================================

all_test_trades = (

    pd.concat(
        test_trade_frames
    )

    .sort_index()

)


overall_fixed = return_stats(

    all_test_trades[
        "fixed_return"
    ]

)


overall_sized = return_stats(

    all_test_trades[
        "sized_return"
    ]

)


print()
print("=" * 80)

print(
    "OVERALL OOS POSITION SIZING"
)

print("=" * 80)


print()
print(
    "FIXED"
)

print(
    overall_fixed
)


print()
print(
    "SIZED"
)

print(
    overall_sized
)


# ============================================================
# 26. Confidence × Size × Profit
# ============================================================

POSITION_CONF_BINS = [

    0.50,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    1.000001,

]


POSITION_CONF_LABELS = [

    "50-54",
    "54-56",
    "56-58",
    "58-60",
    "60-62",
    "62-65",
    "65-70",
    "70+",

]


all_test_trades[
    "confidence_band"
] = pd.cut(

    all_test_trades[
        "confidence"
    ],

    bins=
        POSITION_CONF_BINS,

    labels=
        POSITION_CONF_LABELS,

    right=False,

)


confidence_rows = []


for band, group in (

    all_test_trades

    .groupby(
        "confidence_band",
        observed=True
    )

):


    if len(
        group
    ) == 0:

        continue


    fixed = return_stats(

        group[
            "fixed_return"
        ]

    )


    sized = return_stats(

        group[
            "sized_return"
        ]

    )


    confidence_rows.append({

        "confidence_band":
            str(
                band
            ),

        "trades":
            len(
                group
            ),

        "mean_confidence":
            group[
                "confidence"
            ].mean(),

        "mean_size":
            group[
                "position_size"
            ].mean(),

        "fixed_avg":
            fixed[
                "avg_return"
            ],

        "sized_avg":
            sized[
                "avg_return"
            ],

        "fixed_pf":
            fixed[
                "profit_factor"
            ],

        "sized_pf":
            sized[
                "profit_factor"
            ],

    })


confidence_table = pd.DataFrame(

    confidence_rows

)


print()
print("=" * 80)

print(
    "CONFIDENCE x POSITION SIZE x PROFIT"
)

print("=" * 80)


confidence_show = (

    confidence_table.copy()

)


for col in [

    "mean_confidence",
    "fixed_avg",
    "sized_avg",

]:

    confidence_show[
        col
    ] *= 100


print(

    confidence_show.to_string(
        index=False
    )

)


# ============================================================
# 27. Moving Block Bootstrap
#
# 目的:
#
# Sizing - Fixed
#
# の利益差が偶然か確認する。
#
# 30分HoldなのでTrade同士に依存性がある可能性がある。
#
# そのため、
# trade単位の単純Bootstrapではなく
# 日次に集約してBlock Bootstrapする。
# ============================================================

bootstrap_source = (

    all_test_trades[
        [

            "fixed_return",
            "sized_return",

        ]
    ]

    .resample(
        "1D"
    )

    .sum()

)


bootstrap_source[
    "delta"
] = (

    bootstrap_source[
        "sized_return"
    ]

    -

    bootstrap_source[
        "fixed_return"
    ]

)


daily_delta = (

    bootstrap_source[
        "delta"
    ]

    .to_numpy(
        dtype=float
    )

)


# ------------------------------------------------------------
# 完全ゼロDayも時間構造なので残す
# ------------------------------------------------------------

n_days = len(
    daily_delta
)


rng = np.random.default_rng(

    RANDOM_SEED

)


bootstrap_means = []


if (
    n_days
    >
    BOOTSTRAP_BLOCK_DAYS
):


    max_start = (

        n_days

        -

        BOOTSTRAP_BLOCK_DAYS

    )


    number_blocks = int(

        np.ceil(

            n_days

            /

            BOOTSTRAP_BLOCK_DAYS

        )

    )


    for _ in range(

        BOOTSTRAP_ITERATIONS

    ):


        sample_parts = []


        for __ in range(

            number_blocks

        ):


            start = rng.integers(

                0,

                max_start + 1

            )


            block = daily_delta[

                start
                :
                start
                +
                BOOTSTRAP_BLOCK_DAYS

            ]


            sample_parts.append(
                block
            )


        sample = np.concatenate(

            sample_parts

        )[:n_days]


        bootstrap_means.append(

            np.mean(
                sample
            )

        )


bootstrap_means = np.asarray(

    bootstrap_means,

    dtype=float,

)


if len(
    bootstrap_means
) > 0:


    ci_low = np.percentile(

        bootstrap_means,
        2.5

    )


    ci_high = np.percentile(

        bootstrap_means,
        97.5

    )


    prob_positive = np.mean(

        bootstrap_means
        >
        0

    )


else:

    ci_low = np.nan

    ci_high = np.nan

    prob_positive = np.nan


print()
print("=" * 80)

print(
    "POSITION SIZING BLOCK BOOTSTRAP"
)

print("=" * 80)


print(
    "Observed mean daily improvement:",
    np.mean(
        daily_delta
    )
    *
    100,
    "%"
)


print(
    "95% CI:",
    ci_low
    *
    100,
    "%",
    "~",
    ci_high
    *
    100,
    "%"
)


print(
    "Probability improvement > 0:",
    prob_positive
    *
    100,
    "%"
)


# ============================================================
# 28. Automatic Decision
# ============================================================

print()
print("=" * 80)

print(
    "AUTOMATIC DECISION"
)

print("=" * 80)


print(
    "Fixed PF:",
    overall_fixed[
        "profit_factor"
    ]
)


print(
    "Sized PF:",
    overall_sized[
        "profit_factor"
    ]
)


print()


print(
    "Fixed Avg Return:",
    overall_fixed[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Sized Avg Return:",
    overall_sized[
        "avg_return"
    ]
    *
    100,
    "%"
)


print()


print(
    "Fixed Max DD:",
    overall_fixed[
        "max_dd"
    ]
    *
    100,
    "%"
)


print(
    "Sized Max DD:",
    overall_sized[
        "max_dd"
    ]
    *
    100,
    "%"
)


print()


print(
    "Avg better years:",
    annual_results[
        "avg_better"
    ].sum(),
    "/",
    n_years
)


print(
    "PF better years:",
    annual_results[
        "pf_better"
    ].sum(),
    "/",
    n_years
)


print(
    "Return/DD better years:",
    annual_results[
        "return_dd_better"
    ].sum(),
    "/",
    n_years
)


print(
    "Bootstrap P(delta > 0):",
    prob_positive
)


# ============================================================
# 29. 判定
# ============================================================

clear_value = (

    overall_sized[
        "profit_factor"
    ]

    >

    overall_fixed[
        "profit_factor"
    ]

    and

    overall_sized[
        "avg_return"
    ]

    >

    overall_fixed[
        "avg_return"
    ]

    and

    annual_results[
        "avg_better"
    ].sum()

    >=

    max(
        4,
        int(
            np.ceil(
                n_years
                *
                0.55
            )
        )
    )

    and

    np.isfinite(
        prob_positive
    )

    and

    prob_positive
    >=
    0.95

)


print()


if clear_value:

    print(
        "判定: POSITION SIZING HAS CLEAR OOS VALUE"
    )

    print()

    print(
        "→ Confidence-based Position Sizingを正式候補にします。"
    )

else:

    print(
        "判定: POSITION SIZING VALUE IS NOT YET CLEAR"
    )

    print()

    print(
        "→ FIXED 1.0xを維持します。"
    )

    print(
        "→ Sizing候補を増やして無理に最適化しません。"
    )


# ============================================================
# 30. Equity Curve
# ============================================================

fixed_equity = (

    1.0
    +
    all_test_trades[
        "fixed_return"
    ]

).cumprod()


sized_equity = (

    1.0
    +
    all_test_trades[
        "sized_return"
    ]

).cumprod()


plt.figure(

    figsize=(
        10,
        6
    )

)


plt.plot(

    fixed_equity.index,

    fixed_equity.values,

    label="Fixed 1.0x",

)


plt.plot(

    sized_equity.index,

    sized_equity.values,

    label="Confidence Sizing",

)


plt.xlabel(
    "Date"
)


plt.ylabel(
    "Equity"
)


plt.title(
    "Nested OOS Position Sizing"
)


plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 31. Position Size distribution
# ============================================================

plt.figure(

    figsize=(
        8,
        5
    )

)


plt.hist(

    all_test_trades[
        "position_size"
    ],

    bins=20,

)


plt.xlabel(
    "Relative Position Size"
)


plt.ylabel(
    "Trades"
)


plt.title(
    "OOS Position Size Distribution"
)


plt.tight_layout()

plt.show()


# ============================================================
# 32. Save
# ============================================================

annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_position_sizing.csv",

    index=False,

)


all_test_trades.to_csv(

    OUTPUT_DIR
    /
    "oos_position_sizing_trades.csv",

)


confidence_table.to_csv(

    OUTPUT_DIR
    /
    "confidence_position_profit.csv",

    index=False,

)


bootstrap_source.to_csv(

    OUTPUT_DIR
    /
    "daily_position_sizing_delta.csv",

)


if validation_policy_tables:

    pd.concat(

        validation_policy_tables,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR
        /
        "validation_sizing_search.csv",

        index=False,

    )


pd.DataFrame({

    "bootstrap_mean_delta":
        bootstrap_means

}).to_csv(

    OUTPUT_DIR
    /
    "bootstrap_distribution.csv",

    index=False,

)


print()
print("=" * 80)

print(
    "FINISHED"
)

print("=" * 80)


print(

    OUTPUT_DIR.resolve()

)


print()
print(
    "次にスクショしてほしい場所:"
)

print(
    "1. ANNUAL POSITION SIZING RESULTS"
)

print(
    "2. SIZING POLICY SELECTION FREQUENCY"
)

print(
    "3. POSITION SIZING YEAR STABILITY"
)

print(
    "4. OVERALL OOS POSITION SIZING"
)

print(
    "5. CONFIDENCE x POSITION SIZE x PROFIT"
)

print(
    "6. POSITION SIZING BLOCK BOOTSTRAP"
)

print(
    "7. AUTOMATIC DECISION"
)


## 元セルindex 45


In [ ]:
# ============================================================
# USD/JPY
# POSITION SIZING
# EXPOSURE DECOMPOSITION / ALLOCATION ALPHA TEST
#
# 目的:
#
# Adaptive Position Sizingの改善が
#
#   1. 単純に平均Position Sizeが増えた効果
#
# なのか、
#
#   2. 高Confidence取引へExposureを配分した効果
#
# なのかを切り分ける。
#
#
# 比較:
#
# FIXED
#   全取引 1.0x
#
# EQUAL_EXPOSURE
#   各年のAdaptive平均Sizeを
#   その年の全取引へ均等適用
#
# ADAPTIVE
#   実際に選択されたConfidence Sizing
#
#
# 注意:
# EQUAL_EXPOSUREは診断用。
# 本番売買戦略ではない。
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. 前回結果の確認
# ============================================================

REQUIRED_COLUMNS = [
    "fixed_return",
    "sized_return",
    "position_size",
    "test_year",
    "sizing_policy",
]


if "all_test_trades" not in globals():

    raise RuntimeError(
        "前回のPosition Sizingコードを先に実行してください。\n"
        "all_test_trades がありません。"
    )


missing_columns = [

    col

    for col in REQUIRED_COLUMNS

    if col not in all_test_trades.columns

]


if missing_columns:

    raise RuntimeError(
        "all_test_trades に必要な列がありません。\n"
        f"不足: {missing_columns}"
    )


# ============================================================
# 1. 設定
# ============================================================

BOOTSTRAP_ITERATIONS = 10000

BOOTSTRAP_BLOCK_DAYS = 20

RANDOM_SEED = 42


OUTPUT_DIR = (

    Path.cwd()

    /

    (
        "position_sizing_exposure_decomposition_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. データ準備
# ============================================================

trades = (

    all_test_trades
    .copy()
    .sort_index()

)


# ------------------------------------------------------------
# indexがDatetimeIndexでなければ変換
# ------------------------------------------------------------

if not isinstance(
    trades.index,
    pd.DatetimeIndex
):

    trades.index = pd.to_datetime(
        trades.index
    )


# ------------------------------------------------------------
# 数値列
# ------------------------------------------------------------

for col in [

    "fixed_return",
    "sized_return",
    "position_size",

]:

    trades[col] = pd.to_numeric(
        trades[col],
        errors="coerce"
    )


trades = trades.dropna(
    subset=[
        "fixed_return",
        "sized_return",
        "position_size",
        "test_year",
    ]
)


trades["test_year"] = (
    trades["test_year"]
    .astype(int)
)


# ============================================================
# 3. Return Stats
# ============================================================

def calc_stats(
    returns
):

    """
    リターン系列を評価する。

    returnはdecimal形式。

    0.001 = +0.1%
    """

    r = (

        pd.Series(
            returns
        )

        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan
        )

        .dropna()

        .astype(float)

        .to_numpy()

    )


    n = len(
        r
    )


    if n == 0:

        return {

            "trades": 0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "growth":
                np.nan,

            "max_dd":
                np.nan,

            "trade_sharpe":
                np.nan,

            "return_to_dd":
                np.nan,

        }


    # --------------------------------------------------------
    # Win Rate
    # --------------------------------------------------------

    win_rate = float(
        np.mean(
            r > 0
        )
    )


    # --------------------------------------------------------
    # Profit Factor
    # --------------------------------------------------------

    gross_profit = float(
        r[
            r > 0
        ].sum()
    )


    gross_loss = float(
        -r[
            r < 0
        ].sum()
    )


    if gross_loss > 0:

        pf = (
            gross_profit
            /
            gross_loss
        )

    elif gross_profit > 0:

        pf = np.inf

    else:

        pf = np.nan


    # --------------------------------------------------------
    # Equity Curve
    # --------------------------------------------------------

    equity = np.cumprod(
        1.0 + r
    )


    equity_with_start = np.concatenate(
        [
            [1.0],
            equity,
        ]
    )


    peak = np.maximum.accumulate(
        equity_with_start
    )


    drawdown = (

        equity_with_start
        /
        peak
        -
        1.0

    )


    max_dd = float(
        np.min(
            drawdown
        )
    )


    growth = float(
        equity[-1]
        -
        1.0
    )


    # --------------------------------------------------------
    # Trade Sharpe
    #
    # 年率Sharpeではない。
    # 同一データ内の戦略比較用。
    # --------------------------------------------------------

    std = np.std(
        r,
        ddof=1
    )


    if (
        np.isfinite(std)
        and
        std > 0
    ):

        trade_sharpe = (

            np.mean(r)
            /
            std
            *
            np.sqrt(n)

        )

    else:

        trade_sharpe = np.nan


    # --------------------------------------------------------
    # Growth / DD
    # --------------------------------------------------------

    if max_dd < 0:

        return_to_dd = (

            growth
            /
            abs(max_dd)

        )

    else:

        return_to_dd = np.nan


    return {

        "trades":
            n,

        "win_rate":
            win_rate,

        "avg_return":
            float(
                np.mean(r)
            ),

        "median_return":
            float(
                np.median(r)
            ),

        "profit_factor":
            float(pf),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "trade_sharpe":
            float(
                trade_sharpe
            ),

        "return_to_dd":
            float(
                return_to_dd
            ),

    }


# ============================================================
# 4. 年ごとの平均Exposureを計算
# ============================================================

year_exposure = (

    trades

    .groupby(
        "test_year"
    )[
        "position_size"
    ]

    .mean()

    .rename(
        "year_mean_size"
    )

)


trades = trades.join(

    year_exposure,

    on="test_year"

)


# ============================================================
# 5. Equal Exposure Control
# ============================================================

# ------------------------------------------------------------
# Adaptiveがその年平均で1.11倍だった場合、
#
# Fixed Return × 1.11
#
# とする。
#
# これが
#
# 「Confidenceを無視して全部同じだけレバレッジを増やした場合」
#
# ------------------------------------------------------------

trades[
    "equal_exposure_return"
] = (

    trades[
        "fixed_return"
    ]

    *

    trades[
        "year_mean_size"
    ]

)


# ============================================================
# 6. Exposure Normalized Adaptive
# ============================================================

# ------------------------------------------------------------
# Adaptiveを各年平均Exposure=1へ戻す。
#
# これでもFixedを上回れば、
#
# 「総Exposureを増やしたから」
#
# では説明できない。
# ------------------------------------------------------------

trades[
    "normalized_adaptive_return"
] = (

    trades[
        "sized_return"
    ]

    /

    trades[
        "year_mean_size"
    ]

)


# ============================================================
# 7. Allocation Alpha
# ============================================================

# ------------------------------------------------------------
# 同じ平均Exposureならどちらが強いか。
#
#
# Adaptive - Equal Exposure
#
#
# > 0
#
# なら、
#
# Confidenceに応じた配分そのものが価値を出した。
# ------------------------------------------------------------

trades[
    "allocation_alpha"
] = (

    trades[
        "sized_return"
    ]

    -

    trades[
        "equal_exposure_return"
    ]

)


# ============================================================
# 8. 年別分析
# ============================================================

annual_rows = []


for year, group in trades.groupby(
    "test_year"
):


    fixed_stats = calc_stats(
        group[
            "fixed_return"
        ]
    )


    equal_stats = calc_stats(
        group[
            "equal_exposure_return"
        ]
    )


    adaptive_stats = calc_stats(
        group[
            "sized_return"
        ]
    )


    normalized_stats = calc_stats(
        group[
            "normalized_adaptive_return"
        ]
    )


    policy_values = (

        group[
            "sizing_policy"
        ]

        .astype(str)

        .value_counts()

    )


    if len(
        policy_values
    ) > 0:

        policy = (
            policy_values
            .index[0]
        )

    else:

        policy = "UNKNOWN"


    mean_size = float(
        group[
            "position_size"
        ].mean()
    )


    allocation_alpha_sum = float(
        group[
            "allocation_alpha"
        ].sum()
    )


    allocation_alpha_avg = float(
        group[
            "allocation_alpha"
        ].mean()
    )


    annual_rows.append({

        "test_year":
            int(year),

        "sizing_policy":
            policy,

        "trades":
            len(group),

        "mean_size":
            mean_size,

        "exposure_drift":
            mean_size
            -
            1.0,

        # FIXED
        "fixed_avg":
            fixed_stats[
                "avg_return"
            ],

        "fixed_pf":
            fixed_stats[
                "profit_factor"
            ],

        "fixed_growth":
            fixed_stats[
                "growth"
            ],

        "fixed_dd":
            fixed_stats[
                "max_dd"
            ],

        "fixed_sharpe":
            fixed_stats[
                "trade_sharpe"
            ],

        # Equal Exposure
        "equal_avg":
            equal_stats[
                "avg_return"
            ],

        "equal_pf":
            equal_stats[
                "profit_factor"
            ],

        "equal_growth":
            equal_stats[
                "growth"
            ],

        "equal_dd":
            equal_stats[
                "max_dd"
            ],

        "equal_sharpe":
            equal_stats[
                "trade_sharpe"
            ],

        # Adaptive
        "adaptive_avg":
            adaptive_stats[
                "avg_return"
            ],

        "adaptive_pf":
            adaptive_stats[
                "profit_factor"
            ],

        "adaptive_growth":
            adaptive_stats[
                "growth"
            ],

        "adaptive_dd":
            adaptive_stats[
                "max_dd"
            ],

        "adaptive_sharpe":
            adaptive_stats[
                "trade_sharpe"
            ],

        # Exposure normalized
        "normalized_avg":
            normalized_stats[
                "avg_return"
            ],

        "normalized_pf":
            normalized_stats[
                "profit_factor"
            ],

        "normalized_growth":
            normalized_stats[
                "growth"
            ],

        "normalized_dd":
            normalized_stats[
                "max_dd"
            ],

        "normalized_sharpe":
            normalized_stats[
                "trade_sharpe"
            ],

        # Allocation Alpha
        "allocation_alpha_avg":
            allocation_alpha_avg,

        "allocation_alpha_sum":
            allocation_alpha_sum,

    })


annual = pd.DataFrame(
    annual_rows
)


# ============================================================
# 9. 年別表示
# ============================================================

print()
print("=" * 90)

print(
    "YEARLY EXPOSURE DECOMPOSITION"
)

print("=" * 90)


annual_show = annual.copy()


percentage_columns = [

    "exposure_drift",

    "fixed_avg",
    "equal_avg",
    "adaptive_avg",
    "normalized_avg",

    "fixed_growth",
    "equal_growth",
    "adaptive_growth",
    "normalized_growth",

    "fixed_dd",
    "equal_dd",
    "adaptive_dd",
    "normalized_dd",

    "allocation_alpha_avg",
    "allocation_alpha_sum",

]


for col in percentage_columns:

    annual_show[
        col
    ] *= 100


print(

    annual_show[

        [

            "test_year",

            "sizing_policy",

            "trades",

            "mean_size",

            "exposure_drift",

            "fixed_avg",

            "equal_avg",

            "adaptive_avg",

            "normalized_avg",

            "fixed_pf",

            "equal_pf",

            "adaptive_pf",

            "normalized_pf",

            "allocation_alpha_sum",

        ]

    ].to_string(
        index=False
    )

)


# ============================================================
# 10. 全体OOS
# ============================================================

fixed_overall = calc_stats(
    trades[
        "fixed_return"
    ]
)


equal_overall = calc_stats(
    trades[
        "equal_exposure_return"
    ]
)


adaptive_overall = calc_stats(
    trades[
        "sized_return"
    ]
)


normalized_overall = calc_stats(
    trades[
        "normalized_adaptive_return"
    ]
)


print()
print("=" * 90)

print(
    "OVERALL EXPOSURE DECOMPOSITION"
)

print("=" * 90)


overall_table = pd.DataFrame({

    "FIXED":
        fixed_overall,

    "EQUAL_EXPOSURE":
        equal_overall,

    "ADAPTIVE":
        adaptive_overall,

    "ADAPTIVE_NORMALIZED":
        normalized_overall,

}).T


overall_show = (
    overall_table.copy()
)


for col in [

    "win_rate",
    "avg_return",
    "median_return",
    "growth",
    "max_dd",

]:

    overall_show[
        col
    ] *= 100


print(
    overall_show.to_string()
)


# ============================================================
# 11. Sizingが実際に使われた年だけ分析
# ============================================================

active_annual = annual.loc[

    annual[
        "sizing_policy"
    ]
    !=
    "FIXED"

].copy()


print()
print("=" * 90)

print(
    "ACTIVE SIZING YEARS ONLY"
)

print("=" * 90)


print(
    "Active sizing years:",
    len(
        active_annual
    )
)


if len(
    active_annual
) > 0:


    active_annual[
        "alpha_positive"
    ] = (

        active_annual[
            "allocation_alpha_sum"
        ]
        >
        0

    )


    active_annual[
        "pf_better_than_fixed"
    ] = (

        active_annual[
            "adaptive_pf"
        ]

        >

        active_annual[
            "fixed_pf"
        ]

    )


    active_annual[
        "normalized_avg_better"
    ] = (

        active_annual[
            "normalized_avg"
        ]

        >

        active_annual[
            "fixed_avg"
        ]

    )


    print()

    print(
        active_annual[
            [

                "test_year",

                "sizing_policy",

                "mean_size",

                "fixed_pf",

                "adaptive_pf",

                "fixed_avg",

                "normalized_avg",

                "allocation_alpha_sum",

                "alpha_positive",

                "pf_better_than_fixed",

                "normalized_avg_better",

            ]
        ].to_string(
            index=False
        )
    )


    print()

    print(
        "Allocation Alpha positive:",
        active_annual[
            "alpha_positive"
        ].sum(),
        "/",
        len(
            active_annual
        )
    )


    print(
        "PF better:",
        active_annual[
            "pf_better_than_fixed"
        ].sum(),
        "/",
        len(
            active_annual
        )
    )


    print(
        "Exposure-normalized Avg better:",
        active_annual[
            "normalized_avg_better"
        ].sum(),
        "/",
        len(
            active_annual
        )
    )


# ============================================================
# 12. ConfidenceごとのAllocation Contribution
# ============================================================

CONF_BINS = [

    0.50,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    1.000001,

]


CONF_LABELS = [

    "50-54",

    "54-56",

    "56-58",

    "58-60",

    "60-62",

    "62-65",

    "65-70",

    "70+",

]


if "confidence" in trades.columns:


    trades[
        "confidence_band"
    ] = pd.cut(

        trades[
            "confidence"
        ],

        bins=
            CONF_BINS,

        labels=
            CONF_LABELS,

        right=False,

        include_lowest=True,

    )


    confidence_rows = []


    for band, group in trades.groupby(

        "confidence_band",

        observed=True,

    ):


        if len(
            group
        ) == 0:

            continue


        confidence_rows.append({

            "confidence_band":
                str(
                    band
                ),

            "trades":
                len(
                    group
                ),

            "mean_confidence":
                group[
                    "confidence"
                ].mean(),

            "mean_size":
                group[
                    "position_size"
                ].mean(),

            "fixed_avg":
                group[
                    "fixed_return"
                ].mean(),

            "equal_avg":
                group[
                    "equal_exposure_return"
                ].mean(),

            "adaptive_avg":
                group[
                    "sized_return"
                ].mean(),

            "allocation_alpha_avg":
                group[
                    "allocation_alpha"
                ].mean(),

            "allocation_alpha_sum":
                group[
                    "allocation_alpha"
                ].sum(),

        })


    confidence_table = pd.DataFrame(
        confidence_rows
    )


    print()
    print("=" * 90)

    print(
        "CONFIDENCE x ALLOCATION CONTRIBUTION"
    )

    print("=" * 90)


    confidence_show = (
        confidence_table.copy()
    )


    for col in [

        "mean_confidence",

        "fixed_avg",

        "equal_avg",

        "adaptive_avg",

        "allocation_alpha_avg",

        "allocation_alpha_sum",

    ]:

        confidence_show[
            col
        ] *= 100


    print(
        confidence_show.to_string(
            index=False
        )
    )


else:

    confidence_table = (
        pd.DataFrame()
    )


# ============================================================
# 13. Moving Block Bootstrap
# ============================================================

def moving_block_bootstrap(
    values,
    block_size=20,
    iterations=10000,
    seed=42,
):

    """
    時系列依存をある程度残すための
    Moving Block Bootstrap。
    """

    x = np.asarray(
        values,
        dtype=float
    )


    x = x[
        np.isfinite(
            x
        )
    ]


    n = len(
        x
    )


    if n < block_size:

        return {
            "observed":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_positive":
                np.nan,

            "distribution":
                np.array([]),
        }


    rng = np.random.default_rng(
        seed
    )


    max_start = (
        n
        -
        block_size
    )


    blocks_needed = int(
        np.ceil(
            n
            /
            block_size
        )
    )


    samples = []


    for _ in range(
        iterations
    ):


        pieces = []


        for __ in range(
            blocks_needed
        ):


            start = int(
                rng.integers(
                    0,
                    max_start + 1
                )
            )


            pieces.append(

                x[
                    start
                    :
                    start
                    +
                    block_size
                ]

            )


        sampled = np.concatenate(
            pieces
        )[:n]


        samples.append(
            np.mean(
                sampled
            )
        )


    samples = np.asarray(
        samples
    )


    return {

        "observed":
            float(
                np.mean(x)
            ),

        "ci_low":
            float(
                np.percentile(
                    samples,
                    2.5
                )
            ),

        "ci_high":
            float(
                np.percentile(
                    samples,
                    97.5
                )
            ),

        "prob_positive":
            float(
                np.mean(
                    samples > 0
                )
            ),

        "distribution":
            samples,

    }


# ============================================================
# 14. 日次Allocation Alpha
# ============================================================

daily = (

    trades[
        [
            "allocation_alpha",
        ]
    ]

    .resample(
        "1D"
    )

    .sum()

)


overall_bootstrap = moving_block_bootstrap(

    daily[
        "allocation_alpha"
    ],

    block_size=
        BOOTSTRAP_BLOCK_DAYS,

    iterations=
        BOOTSTRAP_ITERATIONS,

    seed=
        RANDOM_SEED,

)


print()
print("=" * 90)

print(
    "ALLOCATION ALPHA BLOCK BOOTSTRAP"
)

print("=" * 90)


print(
    "Observed Daily Allocation Alpha:",
    overall_bootstrap[
        "observed"
    ]
    *
    100,
    "%"
)


print(
    "95% CI:",
    overall_bootstrap[
        "ci_low"
    ]
    *
    100,
    "%",
    "~",
    overall_bootstrap[
        "ci_high"
    ]
    *
    100,
    "%"
)


print(
    "P(Allocation Alpha > 0):",
    overall_bootstrap[
        "prob_positive"
    ]
    *
    100,
    "%"
)


# ============================================================
# 15. Active Sizing期間のみBootstrap
# ============================================================

active_years = (

    annual.loc[
        annual[
            "sizing_policy"
        ]
        !=
        "FIXED",
        "test_year"
    ]

    .tolist()

)


active_trades = trades.loc[

    trades[
        "test_year"
    ].isin(
        active_years
    )

].copy()


if len(
    active_trades
) > 0:


    active_daily = (

        active_trades[
            [
                "allocation_alpha",
            ]
        ]

        .resample(
            "1D"
        )

        .sum()

    )


    active_bootstrap = moving_block_bootstrap(

        active_daily[
            "allocation_alpha"
        ],

        block_size=
            BOOTSTRAP_BLOCK_DAYS,

        iterations=
            BOOTSTRAP_ITERATIONS,

        seed=
            RANDOM_SEED + 1,

    )


    print()
    print("=" * 90)

    print(
        "ACTIVE YEARS ALLOCATION BOOTSTRAP"
    )

    print("=" * 90)


    print(
        "Active years:",
        active_years
    )


    print(
        "Observed Alpha:",
        active_bootstrap[
            "observed"
        ]
        *
        100,
        "%"
    )


    print(
        "95% CI:",
        active_bootstrap[
            "ci_low"
        ]
        *
        100,
        "%",
        "~",
        active_bootstrap[
            "ci_high"
        ]
        *
        100,
        "%"
    )


    print(
        "P(Alpha > 0):",
        active_bootstrap[
            "prob_positive"
        ]
        *
        100,
        "%"
    )


else:


    active_bootstrap = {

        "observed":
            np.nan,

        "ci_low":
            np.nan,

        "ci_high":
            np.nan,

        "prob_positive":
            np.nan,

        "distribution":
            np.array([]),

    }


# ============================================================
# 16. Automatic Diagnosis
# ============================================================

print()
print("=" * 90)

print(
    "AUTOMATIC DIAGNOSIS"
)

print("=" * 90)


print(
    "Fixed PF:",
    fixed_overall[
        "profit_factor"
    ]
)


print(
    "Equal Exposure PF:",
    equal_overall[
        "profit_factor"
    ]
)


print(
    "Adaptive PF:",
    adaptive_overall[
        "profit_factor"
    ]
)


print(
    "Normalized Adaptive PF:",
    normalized_overall[
        "profit_factor"
    ]
)


print()


print(
    "Fixed Avg Return:",
    fixed_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Equal Exposure Avg Return:",
    equal_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Adaptive Avg Return:",
    adaptive_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Normalized Adaptive Avg Return:",
    normalized_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print()


print(
    "Allocation Bootstrap P:",
    overall_bootstrap[
        "prob_positive"
    ]
)


print(
    "Allocation 95% CI Low:",
    overall_bootstrap[
        "ci_low"
    ]
)


# ============================================================
# 17. 判定条件
# ============================================================

condition_pf = (

    adaptive_overall[
        "profit_factor"
    ]

    >

    fixed_overall[
        "profit_factor"
    ]

)


condition_normalized_return = (

    normalized_overall[
        "avg_return"
    ]

    >

    fixed_overall[
        "avg_return"
    ]

)


condition_bootstrap = (

    np.isfinite(
        overall_bootstrap[
            "prob_positive"
        ]
    )

    and

    overall_bootstrap[
        "prob_positive"
    ]
    >=
    0.95

)


condition_ci = (

    np.isfinite(
        overall_bootstrap[
            "ci_low"
        ]
    )

    and

    overall_bootstrap[
        "ci_low"
    ]
    >
    0

)


if len(
    active_annual
) >= 2:


    condition_active_years = (

        (
            active_annual[
                "allocation_alpha_sum"
            ]
            >
            0
        )

        .mean()

        >=
        0.75

    )


else:

    condition_active_years = False


print()
print(
    "PF improvement:",
    condition_pf
)


print(
    "Exposure-normalized Return improvement:",
    condition_normalized_return
)


print(
    "Bootstrap >= 95%:",
    condition_bootstrap
)


print(
    "Bootstrap CI lower > 0:",
    condition_ci
)


print(
    "Active sizing year stability:",
    condition_active_years
)


# ============================================================
# 18. Final Decision
# ============================================================

clear_allocation_value = (

    condition_pf

    and

    condition_normalized_return

    and

    condition_bootstrap

    and

    condition_ci

    and

    condition_active_years

)


print()
print("=" * 90)

print(
    "FINAL DECISION"
)

print("=" * 90)


if clear_allocation_value:


    print(
        "RESULT: CONFIDENCE ALLOCATION HAS CLEAR OOS VALUE"
    )

    print()

    print(
        "単純なExposure増加だけでは改善を説明できません。"
    )

    print(
        "Confidence-based Adaptive Position Sizingを"
    )

    print(
        "正式な戦略候補として固定できます。"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Exposure / Risk Engineの検証へ進みます。"
    )


else:


    print(
        "RESULT: ALLOCATION VALUE IS NOT YET CLEAR"
    )

    print()

    print(
        "Position Sizingの正式採用は保留します。"
    )

    print(
        "FIXED 1.0xを基準として維持します。"
    )

    print()

    print(
        "Sizing係数を結果に合わせて再最適化するのは避けます。"
    )


# ============================================================
# 19. Equity Curve
# ============================================================

fixed_equity = (

    1.0

    +

    trades[
        "fixed_return"
    ]

).cumprod()


equal_equity = (

    1.0

    +

    trades[
        "equal_exposure_return"
    ]

).cumprod()


adaptive_equity = (

    1.0

    +

    trades[
        "sized_return"
    ]

).cumprod()


normalized_equity = (

    1.0

    +

    trades[
        "normalized_adaptive_return"
    ]

).cumprod()


plt.figure(
    figsize=(
        11,
        6
    )
)


plt.plot(
    fixed_equity.index,
    fixed_equity.values,
    label="Fixed"
)


plt.plot(
    equal_equity.index,
    equal_equity.values,
    label="Equal Exposure"
)


plt.plot(
    adaptive_equity.index,
    adaptive_equity.values,
    label="Adaptive"
)


plt.plot(
    normalized_equity.index,
    normalized_equity.values,
    label="Adaptive Normalized"
)


plt.xlabel(
    "Date"
)


plt.ylabel(
    "Equity"
)


plt.title(
    "Exposure Decomposition"
)


plt.legend()


plt.tight_layout()


plt.savefig(
    OUTPUT_DIR
    /
    "exposure_decomposition_equity.png",
    dpi=150,
)


plt.close()


# ============================================================
# 20. Save
# ============================================================

annual.to_csv(

    OUTPUT_DIR
    /
    "annual_exposure_decomposition.csv",

    index=False,

)


trades.to_csv(

    OUTPUT_DIR
    /
    "trade_level_exposure_decomposition.csv",

)


overall_table.to_csv(

    OUTPUT_DIR
    /
    "overall_exposure_decomposition.csv",

)


if not confidence_table.empty:

    confidence_table.to_csv(

        OUTPUT_DIR
        /
        "confidence_allocation_contribution.csv",

        index=False,

    )


daily.to_csv(

    OUTPUT_DIR
    /
    "daily_allocation_alpha.csv",

)


pd.DataFrame({

    "bootstrap_alpha":
        overall_bootstrap[
            "distribution"
        ]

}).to_csv(

    OUTPUT_DIR
    /
    "allocation_bootstrap.csv",

    index=False,

)


print()
print("=" * 90)

print(
    "FINISHED"
)

print("=" * 90)


print(
    OUTPUT_DIR.resolve()
)


print()
print(
    "次にスクショしてほしい場所:"
)


print(
    "1. YEARLY EXPOSURE DECOMPOSITION"
)


print(
    "2. OVERALL EXPOSURE DECOMPOSITION"
)


print(
    "3. ACTIVE SIZING YEARS ONLY"
)


print(
    "4. CONFIDENCE x ALLOCATION CONTRIBUTION"
)


print(
    "5. ALLOCATION ALPHA BLOCK BOOTSTRAP"
)


print(
    "6. ACTIVE YEARS ALLOCATION BOOTSTRAP"
)


print(
    "7. AUTOMATIC DIAGNOSIS"
)


print(
    "8. FINAL DECISION"
)
